# SupeRR Selector — data cleaning

This notebook writes a silver copy of every source file into `silver/`. The original CSVs are left untouched.

Rules taken from the exploration, with the legal-ball choice fixed here:

- A wide, a no-ball, or an other-type extra is an illegal delivery. Other-type extras are re-bowled like no-balls: counting them as legal puts 150 innings past 120 legal balls. Byes, leg-byes, ordinary balls, and the one penalty row stay legal.
- No extra-run or penalty total is added. That amount can be revisited if a later step needs it.
- `innings_ball_num` is the ball sequence.
- `bowl_speed` is dropped.
- Innings 3 and 4 are flagged as super overs.
- Bowler wickets stay separate from run-outs.
- The two obstructing-the-field spellings become one.
- Split seasons keep the earlier year, so the season set is 2022, 2023, 2024, 2025, and 2026.
- The playing list is written twice: a cleaned squad sheet, and a playing-eleven sheet with 11 players per side.

## Load

In [1]:
from pathlib import Path
import shutil

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 40)

def find_data() -> Path:
    start = Path.cwd().resolve()
    for folder in [start, *start.parents]:
        if (folder / "ball_by_ball.csv").exists() and (folder / "competitions.csv").exists():
            return folder
    raise FileNotFoundError("source CSVs were not found from the working directory upward")

DATA = find_data()
SILVER = DATA / "silver"
SILVER.mkdir(exist_ok=True)
print("source:", DATA)
print("silver:", SILVER)

comps = pd.read_csv(DATA / "competitions.csv")
teams = pd.read_csv(DATA / "teams.csv")
matches = pd.read_csv(DATA / "matches.csv")
squads = pd.read_csv(DATA / "competition_team_player_mapping.csv")
appearances = pd.read_csv(DATA / "playing_eleven.csv")
balls = pd.read_csv(DATA / "ball_by_ball.csv")
print("loaded", {name: len(frame) for name, frame in {
    "competitions": comps, "teams": teams, "matches": matches,
    "squads": squads, "appearances": appearances, "balls": balls,
}.items()})

source: D:\6af3166e7b922_data_pack
silver: D:\6af3166e7b922_data_pack\silver


loaded {'competitions': 141, 'teams': 353, 'matches': 883, 'squads': 6413, 'appearances': 28381, 'balls': 200999}


## Competitions

`2022/23` becomes 2022. The same rule covers `2023/24` and `2024/25`. A competition group is added because `match_format` of T20 also contains two non-IPL events.

In [2]:
comps_silver = comps.copy()
comps_silver["season_raw"] = comps_silver["season"].astype(str)
comps_silver["season"] = comps_silver["season_raw"].str.slice(0, 4).astype(int)
comps_silver["comp_group"] = "Other T20"
comps_silver.loc[comps_silver["match_format"].eq("T20I"), "comp_group"] = "T20I"
comps_silver.loc[comps_silver["comp_name"].str.startswith("IPL"), "comp_group"] = "IPL"
comps_silver = comps_silver.drop(columns=["season_raw"])

print("seasons:", sorted(comps_silver["season"].unique().tolist()))
print("competition group")
display(comps_silver["comp_group"].value_counts().rename("competitions").to_frame())
print("split seasons mapped to the earlier year")
display(
    comps.loc[comps["season"].astype(str).str.contains("/"), ["comp_id", "comp_name", "season"]]
    .assign(season_silver=lambda frame: frame["season"].astype(str).str.slice(0, 4).astype(int))
)
comps_silver.to_csv(SILVER / "competitions.csv", index=False)
print("wrote", SILVER / "competitions.csv", "rows", len(comps_silver))

seasons: [2022, 2023, 2024, 2025, 2026]
competition group


,competitions
comp_group,
T20I,134
IPL,5
Other T20,2


split seasons mapped to the earlier year


,comp_id,comp_name,season,season_silver
43,1646,India in New Zealand T20I Series 2022/23,2022/23,2022
44,1599,South Africa in India T20I Series 2022/23,2022/23,2022
45,1608,New Zealand T20I Tri Series 2022/23,2022/23,2022
65,2188,Ireland in Zimbabwe T20I Series 2023/24,2023/24,2023
97,2755,Sri Lanka in New Zealand T20I Series 2024/25,2024/25,2024


wrote D:\6af3166e7b922_data_pack\silver\competitions.csv rows 141


## Teams and squad map

Exploration found one name per team id and one name per player id, with no duplicate keys. These two files are copied unchanged.

In [3]:
for name in ["teams.csv", "competition_team_player_mapping.csv"]:
    shutil.copy2(DATA / name, SILVER / name)
    print("copied", name)

copied teams.csv


copied competition_team_player_mapping.csv


## Matches

The revised-innings columns are ball counts: every filled first-innings value is a multiple of 6 between 30 and 114. The new columns store that length in balls. A blank revised value stays a full 120. When only the first innings has a reduced length, that same length is used for the second innings. Short innings without a revised value are left short, because a side can be bowled out or finish a chase.

In [4]:
matches_silver = matches.copy()
first = matches_silver["first_inn_revised_overs"]
second = matches_silver["second_inn_revised_overs"]
matches_silver["first_innings_balls"] = first.fillna(120).astype(int)
matches_silver["second_innings_balls"] = second.fillna(first).fillna(120).astype(int)
matches_silver["is_reduced"] = first.notna() | second.notna()

print("reduced matches:", int(matches_silver["is_reduced"].sum()))
print("first-innings ball values:", sorted(matches_silver.loc[matches_silver["is_reduced"], "first_innings_balls"].unique().tolist()))
print("second-innings ball values:", sorted(matches_silver.loc[matches_silver["is_reduced"], "second_innings_balls"].unique().tolist()))
matches_silver.to_csv(SILVER / "matches.csv", index=False)
print("wrote", SILVER / "matches.csv", "rows", len(matches_silver))

reduced matches: 34
first-innings ball values: [30, 42, 48, 54, 60, 66, 72, 78, 84, 90, 102, 108, 114, 120]
second-innings ball values: [30, 42, 48, 54, 60, 66, 72, 78, 84, 90, 102, 108, 114]
wrote D:\6af3166e7b922_data_pack\silver\matches.csv rows 883


## Ball by ball

`extras_type` is decoded to a label. `extras1_runs` is the run amount of that extra, and `extras_2_runs` is a second extra on the same ball, such as byes off a no-ball. `is_legal` is false only for a wide, a no-ball (`Overstep`), or an other-type extra, which is re-bowled like a no-ball. `bowl_speed` is removed. Rows are sorted by match, innings, and `innings_ball_num`. The sheet-gap flag is added after the playing list is cleaned, because it depends on that sheet.

In [5]:
EXTRAS = {"None": "ordinary", "Overstep": "no-ball", "wide": "wide", "legbyes": "leg-bye", "byes": "bye", "other": "other", "Penalty": "penalty"}
BOWLER_WICKETS = ["Caught", "Bowled", "LBW", "Stump", "Hit Wicket"]

balls_silver = balls.copy()
balls_silver["extras_label"] = balls_silver["extras_type"].fillna("None").map(EXTRAS)
print("unmapped extras types:", int(balls_silver["extras_label"].isna().sum()))
balls_silver["extras1_runs"] = balls_silver["extras1_runs"].fillna(0).astype(int)
balls_silver["extras_2_runs"] = balls_silver["extras_2_runs"].fillna(0).astype(int)
ILLEGAL = ["wide", "no-ball", "other"]
balls_silver["is_legal"] = ~balls_silver["extras_label"].isin(ILLEGAL)
print("extra runs by label:", balls_silver.groupby("extras_label")["extras1_runs"].sum().to_dict())
odd = balls_silver[
    (balls_silver["extras_label"].eq("ordinary") & balls_silver[["extras1_runs", "extras_2_runs"]].sum(axis=1).gt(0))
    | (balls_silver["extras_label"].isin(["bye", "leg-bye"]) & balls_silver["extras1_runs"].eq(0))
]
print("source rows whose extras amount does not fit the type, kept as supplied:")
print(odd[["match_id", "innings", "innings_ball_num", "extras_type", "runs", "extras1_runs", "extras_2_runs"]].to_string(index=False))
balls_silver["is_super_over"] = balls_silver["innings"].ge(3)
balls_silver["dismissal_type"] = balls_silver["dismissal_type"].replace({"Obstracting the field": "Obstructing the field"})
balls_silver["is_bowler_wicket"] = balls_silver["dismissal_type"].isin(BOWLER_WICKETS)
balls_silver = balls_silver.drop(columns=["bowl_speed"])
balls_silver = balls_silver.sort_values(
    ["match_id", "innings", "innings_ball_num"], kind="mergesort"
).reset_index(drop=True)

print("rows:", f"{len(balls_silver):,}")
print("illegal deliveries (wide, no-ball, or other):", int((~balls_silver["is_legal"]).sum()))
print("super-over deliveries:", int(balls_silver["is_super_over"].sum()))
print("bowler wickets:", int(balls_silver["is_bowler_wicket"].sum()))
print("run outs:", int(balls_silver["dismissal_type"].eq("RunOut").sum()))
print("obstructing spellings:", sorted(balls_silver["dismissal_type"].dropna().unique().tolist()))
print("bowl_speed still present:", "bowl_speed" in balls_silver.columns)

unmapped extras types: 0
extra runs by label: {'bye': 1000, 'leg-bye': 3603, 'no-ball': 576, 'ordinary': 1, 'other': 264, 'penalty': 5, 'wide': 9174}


source rows whose extras amount does not fit the type, kept as supplied:
 match_id  innings  innings_ball_num extras_type  runs  extras1_runs  extras_2_runs
    17555        1                29         NaN     0             0              1
    36097        2                49         NaN     0             1              0
    37099        1                82     legbyes     1             0              0


rows:

 200,999
illegal deliveries (wide, no-ball, or other): 8554
super-over deliveries: 38
bowler wickets: 10272
run outs: 600
obstructing spellings: ['Bowled', 'Caught', 'Hit Wicket', 'LBW', 'Obstructing the field', 'Retire Out', 'RunOut', 'Stump']
bowl_speed still present: False


## Playing list — cleaned squad

One row per match, team, and player. The 40 extra copies agree on every field except batting order, so a player with two orders keeps a blank order and `order_conflict` is true. Captain and keeper are blank on exactly the rows that did not play; those blanks become false.

In [6]:
keys = ["match_id", "team_id", "player_id"]
order_n = appearances.groupby(keys)["batting_order"].transform("nunique")
squad_sheet = appearances.copy()
squad_sheet["order_n"] = order_n
squad_sheet = squad_sheet.drop_duplicates(keys, keep="first").copy()
squad_sheet["order_conflict"] = squad_sheet["order_n"].gt(1)
squad_sheet.loc[squad_sheet["order_conflict"], "batting_order"] = pd.NA

captain_null_is_not_played = squad_sheet["is_captain"].isna().eq(~squad_sheet["is_played"]).all()
keeper_null_is_not_played = squad_sheet["is_wk"].isna().eq(~squad_sheet["is_played"]).all()
print("captain is blank exactly where the player did not play:", bool(captain_null_is_not_played))
print("keeper is blank exactly where the player did not play:", bool(keeper_null_is_not_played))
if not (captain_null_is_not_played and keeper_null_is_not_played):
    raise AssertionError("captain or keeper blanks are not limited to players who did not play")

squad_sheet.loc[~squad_sheet["is_played"], "is_captain"] = False
squad_sheet.loc[~squad_sheet["is_played"], "is_wk"] = False
squad_sheet["is_captain"] = squad_sheet["is_captain"].astype("boolean")
squad_sheet["is_wk"] = squad_sheet["is_wk"].astype("boolean")
squad_sheet["batting_order"] = squad_sheet["batting_order"].astype("Int64")
squad_sheet = squad_sheet.drop(columns=["order_n"])
squad_sheet = squad_sheet.sort_values(
    ["match_id", "team_id", "batting_order"], kind="mergesort", na_position="last"
).reset_index(drop=True)

print("rows before:", len(appearances), "rows after:", len(squad_sheet))
print("duplicate keys after:", int(squad_sheet.duplicated(keys).sum()))
print("order conflicts:", int(squad_sheet["order_conflict"].sum()))
print("captain blanks left:", int(squad_sheet["is_captain"].isna().sum()))
print("keeper blanks left:", int(squad_sheet["is_wk"].isna().sum()))
squad_sheet.to_csv(SILVER / "playing_eleven.csv", index=False)
print("wrote", SILVER / "playing_eleven.csv")

captain is blank exactly where the player did not play: True
keeper is blank exactly where the player did not play: True
rows before: 28381 rows after: 28341
duplicate keys after: 0
order conflicts: 35
captain blanks left: 0
keeper blanks left: 0
wrote D:\6af3166e7b922_data_pack\silver\playing_eleven.csv


## Playing list — eleven per side

A side is kept when batting orders 1 through 11 each belong to one played player. Players numbered 12 or 13 stay on the squad sheet only. A match is kept when both sides pass, so every match in this file has 11 players on each side.

Captain and keeper stay only if every retained side has exactly one. Otherwise the column is dropped, because a missing captain or keeper is a data issue. Impact flags stay: they have no blanks, and they are not a one-per-side role.

In [7]:
xi_pool = squad_sheet.loc[
    squad_sheet["is_played"]
    & ~squad_sheet["order_conflict"]
    & squad_sheet["batting_order"].between(1, 11)
].copy()
side = xi_pool.groupby(["match_id", "team_id"])["batting_order"].agg(n="count", nunq="nunique", lo="min", hi="max")
clean_sides = side[(side["n"] == 11) & (side["nunq"] == 11) & (side["lo"] == 1) & (side["hi"] == 11)].reset_index()
sides_per_match = clean_sides.groupby("match_id")["team_id"].nunique()
both_sides = sides_per_match[sides_per_match.eq(2)].index
one_side = sides_per_match[sides_per_match.eq(1)].index
source_matches = set(squad_sheet["match_id"])
print("source matches on the playing list:", len(source_matches))
print("sides with orders 1-11 once each:", len(clean_sides))
print("matches with both sides clean:", len(both_sides))
print("matches with only one clean side, excluded:", len(one_side))
print("matches with no clean side, excluded:", len(source_matches - set(sides_per_match.index)))

playing_xi = xi_pool.merge(
    clean_sides.loc[clean_sides["match_id"].isin(both_sides), ["match_id", "team_id"]],
    on=["match_id", "team_id"],
    how="inner",
    validate="many_to_one",
)
side_size = playing_xi.groupby(["match_id", "team_id"]).size()
print("players per side:", sorted(side_size.unique().tolist()))
print("sides:", int(side_size.size), "matches:", playing_xi["match_id"].nunique())

role_keep = []
for column in ["is_captain", "is_wk"]:
    marked = playing_xi.loc[playing_xi[column].eq(True)].groupby(["match_id", "team_id"]).size()
    every_side_has_one = marked.eq(1).all() and marked.size == side_size.size
    print(f"{column}: sides with exactly one = {int(marked.eq(1).sum()) if len(marked) else 0} of {int(side_size.size)}")
    if every_side_has_one:
        role_keep.append(column)
    else:
        print(f"dropped {column}: a playing eleven column stays only when every side has exactly one")

keep_columns = ["match_id", "team_id", "player_id", "batting_order", "is_played", *role_keep, "is_impact", "is_impact_off"]
playing_xi = playing_xi[keep_columns].sort_values(
    ["match_id", "team_id", "batting_order"], kind="mergesort"
).reset_index(drop=True)
print("columns kept:", keep_columns)
print("impact flags inside the elevens:", int(playing_xi["is_impact"].eq(1).sum()),
      "impact, ", int(playing_xi["is_impact_off"].eq(1).sum()), "replaced")
playing_xi.to_csv(SILVER / "playing_xi.csv", index=False)
print("wrote", SILVER / "playing_xi.csv", "rows", len(playing_xi))

source matches on the playing list: 880
sides with orders 1-11 once each: 1727
matches with both sides clean: 861
matches with only one clean side, excluded: 5
matches with no clean side, excluded: 14
players per side: [11]
sides: 1722 matches: 861
is_captain: sides with exactly one = 1665 of 1722
dropped is_captain: a playing eleven column stays only when every side has exactly one
is_wk: sides with exactly one = 1656 of 1722
dropped is_wk: a playing eleven column stays only when every side has exactly one
columns kept: ['match_id', 'team_id', 'player_id', 'batting_order', 'is_played', 'is_impact', 'is_impact_off']
impact flags inside the elevens: 327 impact,  385 replaced


wrote D:\6af3166e7b922_data_pack\silver\playing_xi.csv rows 18942


## Ball by ball — sheet gap and write

A delivery is flagged when the striker, the non-striker, or the bowler is absent from that team's cleaned squad sheet. The rows stay in the file.

In [8]:
listed = squad_sheet.loc[:, ["match_id", "team_id", "player_id"]].drop_duplicates()

def listed_on_sheet(player_col, team_col):
    probe = balls_silver[["match_id", team_col, player_col]].merge(
        listed,
        left_on=["match_id", team_col, player_col],
        right_on=["match_id", "team_id", "player_id"],
        how="left",
        indicator=True,
        validate="many_to_one",
    )
    if len(probe) != len(balls_silver):
        raise AssertionError(f"{player_col} lookup changed the delivery count")
    return probe["_merge"].eq("both").to_numpy()

balls_silver["sheet_gap"] = ~(
    listed_on_sheet("striker_id", "batting_team_id")
    & listed_on_sheet("non_striker_id", "batting_team_id")
    & listed_on_sheet("bowler_id", "bowling_team_id")
)
gap_matches = sorted(balls_silver.loc[balls_silver["sheet_gap"], "match_id"].unique().tolist())
print("deliveries with a player missing from the squad sheet:", int(balls_silver["sheet_gap"].sum()))
print("matches:", gap_matches)

balls_silver.to_csv(SILVER / "ball_by_ball.csv", index=False)
print("wrote", SILVER / "ball_by_ball.csv", "rows", len(balls_silver))

deliveries with a player missing from the squad sheet: 175
matches: [37087, 37092, 37095]


wrote D:\6af3166e7b922_data_pack\silver\ball_by_ball.csv rows 200999


## Checks

In [9]:
written = sorted(path.name for path in SILVER.glob("*.csv"))
print("silver files:", written)

check_balls = pd.read_csv(SILVER / "ball_by_ball.csv")
check_xi = pd.read_csv(SILVER / "playing_xi.csv")
check_squad = pd.read_csv(SILVER / "playing_eleven.csv")
check_comps = pd.read_csv(SILVER / "competitions.csv")

sequence = ["match_id", "innings", "innings_ball_num"]
sorted_ok = check_balls[sequence].reset_index(drop=True).equals(
    check_balls[sequence].sort_values(sequence).reset_index(drop=True)
)
per_side = check_xi.groupby(["match_id", "team_id"]).size()
per_match = check_xi.groupby("match_id")["team_id"].nunique()

checks = {
    "ball rows unchanged": len(check_balls) == len(balls),
    "bowl_speed removed": "bowl_speed" not in check_balls.columns,
    "balls sorted by innings_ball_num": bool(sorted_ok),
    "super overs are innings 3 and 4": int(check_balls["is_super_over"].sum()) == int(balls["innings"].ge(3).sum()),
    "every extras type is labelled": bool(check_balls["extras_label"].notna().all()),
    "no wide, no-ball, or other is marked legal": bool((~check_balls.loc[check_balls["extras_label"].isin(ILLEGAL), "is_legal"]).all()),
    "byes, leg-byes, penalty, and ordinary balls stay legal": bool(check_balls.loc[~check_balls["extras_label"].isin(ILLEGAL), "is_legal"].all()),
    "no innings passes 120 legal balls": int(check_balls.loc[~check_balls["is_super_over"]].groupby(["match_id", "innings"])["is_legal"].sum().max()) <= 120,
    "at most two ordinary balls carry extra runs": int(check_balls.loc[check_balls["extras_label"].eq("ordinary"), ["extras1_runs", "extras_2_runs"]].sum(axis=1).gt(0).sum()) <= 2,
    "one obstructing spelling": "Obstracting the field" not in set(check_balls["dismissal_type"].dropna()),
    "squad keys unique": not check_squad.duplicated(["match_id", "team_id", "player_id"]).any(),
    "eleven is 11 per side": bool((per_side == 11).all()),
    "eleven matches have two sides": bool((per_match == 2).all()),
    "seasons are 2022-2026": set(check_comps["season"]) <= {2022, 2023, 2024, 2025, 2026},
}
for name, ok in checks.items():
    print(("ok  " if ok else "FAIL"), name)
if not all(checks.values()):
    raise AssertionError("one or more silver checks failed")
print("silver directory is complete")

silver files: ['ball_by_ball.csv', 'competition_team_player_mapping.csv', 'competitions.csv', 'matches.csv', 'playing_eleven.csv', 'playing_xi.csv', 'teams.csv']


ok   ball rows unchanged
ok   bowl_speed removed
ok   balls sorted by innings_ball_num
ok   super overs are innings 3 and 4
ok   every extras type is labelled
ok   no wide, no-ball, or other is marked legal
ok   byes, leg-byes, penalty, and ordinary balls stay legal
ok   no innings passes 120 legal balls
ok   at most two ordinary balls carry extra runs
ok   one obstructing spelling
ok   squad keys unique
ok   eleven is 11 per side
ok   eleven matches have two sides
ok   seasons are 2022-2026
silver directory is complete
